In [2]:
import os
import time
import re
import math
import heapq
import random
import faiss
from typing import List, Optional
import numpy as np
import plotly.graph_objects as go
from scipy.io import loadmat
from tqdm import tqdm

In [22]:
import numpy as np
import faiss
from tqdm import tqdm

def evaluate_hit_rate_topk_faiss_lsh(faiss_index, index_labels, query_vectors, query_labels,
                                     k_max=50, n_indexed=None):
    """
    Evaluate top-K hit rate for a FAISS LSH index.

    Parameters
    ----------
    faiss_index : faiss.Index
        A FAISS index (expected to be an LSH index created by `faiss.IndexLSH(d, n_bits)` and already `add`ed).
    index_labels : np.ndarray
        1D array of labels corresponding to the indexed vectors (shape: (n_indexed,))
    query_vectors : np.ndarray
        2D array of query vectors (dtype=float32), shape: (n_queries, d)
    query_labels : np.ndarray
        1D array of labels for each query (shape: (n_queries,))
    k_max : int
        Maximum K to evaluate (will be clipped to number of indexed vectors)
    n_indexed : int or None
        Number of vectors actually indexed in faiss_index. If None, inferred from `index_labels`.

    Returns
    -------
    x_labels : list[str]
        Labels for the x-axis like "k (penetration)".
    hit_rates : list[float]
        Hit rates for each k from 1..k_max.
    """
    if n_indexed is None:
        total_indexed = len(index_labels)
    else:
        total_indexed = int(n_indexed)

    if total_indexed == 0 or len(query_labels) == 0:
        raise ValueError("Index or query set empty.")

    k_max = min(k_max, total_indexed)
    hit_rates = []
    x_labels = []

    # FAISS expects float32 contiguous arrays
    query_vectors = np.ascontiguousarray(query_vectors.astype('float32'))

    for k in tqdm(range(1, k_max + 1,10), desc="Evaluating top-K hit rates", position=0):
        hits = 0

        # iterate over queries
        for qvec, qlabel in tqdm(zip(query_vectors, query_labels),
                                 total=len(query_labels),
                                 desc=f"K={k}",
                                 leave=False,
                                 position=1):
            # faiss.Index.search expects a (1, d) array
            D, I = faiss_index.search(qvec.reshape(1, -1), k)

            # I is shape (1, k) with indices or -1 for missing
            retrieved_idxs = I[0].tolist()

            # filter out -1 indices
            retrieved_idxs = [int(ii) for ii in retrieved_idxs if ii != -1]

            if len(retrieved_idxs) == 0:
                # no neighbors returned
                continue

            retrieved_labels = index_labels[retrieved_idxs]
            if np.any(retrieved_labels == qlabel):
                hits += 1

        hit_rate = hits / len(query_labels)
        hit_rates.append(hit_rate)
        penetration = k / total_indexed
        x_labels.append(f"{k} ({penetration:.3f})")

        tqdm.write(f"K={k:3d}, Hit rate={hit_rate:.4f}")

    return x_labels, hit_rates


In [4]:
# =========================================================
#  Plot Hit Rate vs Penetration Rate
# =========================================================
def plot_hit_rate_plotly_topk(penetration_rates, hit_rates, efSearch,title_name="IITD_LSH"):

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=penetration_rates,
        y=hit_rates,
        mode='lines+markers',
        name=f"efSearch={efSearch}",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    # numeric_vals = [
    #     float(re.search(r'\((.*?)\)', str(x)).group(1)) if '(' in str(x) else float(x)
    #     for x in penetration_rates
    # ]
    # max_tick = math.ceil(max(numeric_vals) * 100) / 100  # round up to next 0.01
    # # Clean tick marks at 0.01 intervals
    # tick_vals = np.round(np.arange(0.0, max_tick + 0.001, 0.01), 2).tolist()
    fig.update_xaxes(
        title='Penetration Rate (k / total indexed)',
        # tickvals=tick_vals,
        # tickformat=".2f"
    )
    fig.update_yaxes(title='Hit Rate')

    fig.update_layout(
        title=f"{title_name}",
        template='plotly_white',
        width=800,
        height=500,
        hovermode="x unified"
    )
    fig.show()


# IITDv1

In [5]:
# ---------------------
# Load IITD_UPD Templates (70:30 Split) - as provided by you
# ---------------------
def load_iitd_upd_templates(root_folder, seed=42, split_ratio=0.7):
    """
    For each subject folder (like 1_L, 2_L...), randomly split its .npy files
      - 70% for indexing (gallery/enrollment)
      - 30% for query (probe)
    Returns:
        index_vectors, index_labels, query_vectors, query_labels
    """
    random.seed(seed)
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    subject_folders = sorted(
        [d for d in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, d))]
    )

    for subject in subject_folders:
        subject_path = os.path.join(root_folder, subject)
        files = sorted([f for f in os.listdir(subject_path) if f.endswith('.npy')])

        n = len(files)
        if n == 0:
            print(f"⚠️ Skipping {subject}, no .npy files found.")
            continue

        random.shuffle(files)
        split_point = max(1, int(n * split_ratio))  # At least 1 for indexing
        index_files = files[:split_point]
        query_files = files[split_point:] if split_point < n else []

        # Load index vectors
        for f in index_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            index_vectors.append(vec)
            index_labels.append(subject)

        # Load query vectors
        for f in query_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            query_vectors.append(vec)
            query_labels.append(subject)

        print(f"{subject}: {len(index_files)} index, {len(query_files)} query")

    if len(index_vectors) == 0:
        raise ValueError("No index vectors found. Check `root_folder` and structure.")

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors) if len(query_vectors) > 0 else np.zeros((0, index_vectors[0].shape[0]), dtype='float32'),
        np.array(query_labels)
    )


In [6]:
data_folder = "/home/nishkal/sg/iris_indexing/IITD_UPD"
X_index, y_index, X_query, y_query = load_iitd_upd_templates(data_folder, seed=42)

100_L: 3 index, 2 query
100_R: 3 index, 2 query
101_L: 3 index, 2 query
101_R: 3 index, 2 query
102_L: 3 index, 2 query
102_R: 3 index, 2 query
103_L: 3 index, 2 query
103_R: 3 index, 2 query
104_L: 3 index, 2 query
104_R: 3 index, 2 query
105_L: 3 index, 2 query
105_R: 3 index, 2 query
106_L: 3 index, 2 query
106_R: 3 index, 2 query
107_L: 3 index, 2 query
107_R: 3 index, 2 query
108_L: 3 index, 2 query
108_R: 3 index, 2 query
109_L: 3 index, 2 query
109_R: 3 index, 2 query
10_L: 7 index, 3 query
110_L: 3 index, 2 query
110_R: 3 index, 2 query
111_L: 3 index, 2 query
111_R: 3 index, 2 query
112_L: 3 index, 2 query
112_R: 3 index, 2 query
113_L: 3 index, 2 query
113_R: 3 index, 2 query
114_L: 3 index, 2 query
114_R: 3 index, 2 query
115_L: 3 index, 2 query
115_R: 3 index, 2 query
116_L: 3 index, 2 query
116_R: 3 index, 2 query
117_L: 3 index, 2 query
117_R: 3 index, 2 query
118_L: 3 index, 2 query
118_R: 3 index, 2 query
119_L: 3 index, 2 query
119_R: 3 index, 2 query
11_L: 7 index, 3 

In [8]:
d = X_index.shape[1]
n_bits = 1024                # tune this (bigger -> finer hashing)
index = faiss.IndexLSH(d, n_bits)
index.add(np.ascontiguousarray(X_index.astype('float32')))

x_labels, hit_rates = evaluate_hit_rate_topk_faiss_lsh(
    faiss_index=index,
    index_labels=y_index,
    query_vectors=X_query,
    query_labels=y_query,
    k_max=50
)

Evaluating top-K hit rates:   0%|          | 0/50 [00:00<?, ?it/s]

Evaluating top-K hit rates:   2%|▏         | 1/50 [00:00<00:06,  7.95it/s]

K=  1, Hit rate=0.8607


Evaluating top-K hit rates:   4%|▍         | 2/50 [00:00<00:05,  8.10it/s]

K=  2, Hit rate=0.9083


...

K= 48, Hit rate=0.9841


Evaluating top-K hit rates:  98%|█████████▊| 49/50 [00:06<00:00,  8.05it/s]

K= 49, Hit rate=0.9841


Evaluating top-K hit rates: 100%|██████████| 50/50 [00:06<00:00,  8.08it/s]

K= 50, Hit rate=0.9853


In [9]:
np.savez('IITDv1_LSH.npz', x=x_labels, y=hit_rates)

In [9]:
plot_hit_rate_plotly_topk(x_labels, hit_rates,23,"IITDV1 LSH")

# CASIAv1

In [10]:
def load_CASIAV1_templates(data_folder):
    """
    Load CASIA V1 templates according to the structure:
        <subjectID>_<session>_<sample>.jpg.mat

    Logic:
        - Files ending with '_3.jpg.mat' or '_4.jpg.mat' → Query templates
        - Files ending with '_1.jpg.mat' or '_2.jpg.mat' → Index (enrollment) templates

    Returns:
        index_vectors, index_labels, query_vectors, query_labels
    """
    files = sorted([f for f in os.listdir(data_folder) if f.endswith('.jpg.mat')])
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    for file in files:
        path = os.path.join(data_folder, file)
        data = loadmat(path)
        vec = data['template'].flatten().astype('float32')

        # Extract identifiers
        subject_id, session, sample = file.replace('.jpg.mat', '').split('_')
        label = f"{subject_id}_{session}"  # subject-session pair label

        if sample in ('3', '4'):  # Probe (query)
            query_vectors.append(vec)
            query_labels.append(label)
        else:  # Enrollment (index)
            index_vectors.append(vec)
            index_labels.append(label)

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors),
        np.array(query_labels)
    )

In [11]:
data_folder = "/home/nishkal/sg/iris_indexing/Updated_CASIAV1"
X_index, y_index, X_query, y_query = load_CASIAV1_templates(data_folder)


In [12]:
d = X_index.shape[1]
n_bits = 1024                # tune this (bigger -> finer hashing)
index = faiss.IndexLSH(d, n_bits)
index.add(np.ascontiguousarray(X_index.astype('float32')))

x_labels, hit_rates = evaluate_hit_rate_topk_faiss_lsh(
    faiss_index=index,
    index_labels=y_index,
    query_vectors=X_query,
    query_labels=y_query,
    k_max=50
)


Evaluating top-K hit rates:   2%|▏         | 1/50 [00:00<00:05,  9.63it/s]

K=  1, Hit rate=0.5556


Evaluating top-K hit rates:   2%|▏         | 1/50 [00:00<00:05,  9.63it/s]

K=  2, Hit rate=0.7284


Evaluating top-K hit rates:   6%|▌         | 3/50 [00:00<00:04,  9.99it/s]

...

K= 48, Hit rate=0.9537


Evaluating top-K hit rates:  98%|█████████▊| 49/50 [00:04<00:00, 10.18it/s]

K= 49, Hit rate=0.9537


Evaluating top-K hit rates: 100%|██████████| 50/50 [00:04<00:00, 10.20it/s]

K= 50, Hit rate=0.9537


In [13]:
np.savez('CASIAv1_LSH.npz', x=x_labels, y=hit_rates)

In [12]:
d = X_index.shape[1]
n_bits = 1024                # tune this (bigger -> finer hashing)
index = faiss.IndexLSH(d, n_bits)
index.add(np.ascontiguousarray(X_index.astype('float32')))

x_labels, hit_rates = evaluate_hit_rate_topk_faiss_lsh(
    faiss_index=index,
    index_labels=y_index,
    query_vectors=X_query,
    query_labels=y_query,
    k_max=50
)

plot_hit_rate_plotly_topk(x_labels, hit_rates,23,"CASIAv1 LSH")

Evaluating top-K hit rates:   0%|          | 0/50 [00:00<?, ?it/s]

K=  1, Hit rate=0.5556


Evaluating top-K hit rates:   4%|▍         | 2/50 [00:00<00:03, 15.92it/s]

K=  2, Hit rate=0.7284


Evaluating top-K hit rates:   4%|▍         | 2/50 [00:00<00:03, 15.92it/s]

...

K= 48, Hit rate=0.9537


Evaluating top-K hit rates:  96%|█████████▌| 48/50 [00:03<00:00, 15.55it/s]

K= 49, Hit rate=0.9537


Evaluating top-K hit rates: 100%|██████████| 50/50 [00:03<00:00, 15.88it/s]


K= 50, Hit rate=0.9537


# Iris Lamp

In [14]:

def load_IrisLamp_templates(root_folder, seed=42, pick='random'):
    """
    Process a flat folder of template files named like: 001_L_01_template.npz
    Groups by subject-eye (e.g. '001_L'), picks exactly one file per group for indexing,
    and the rest for querying.

    Args:
        root_folder (str): path to folder containing .npz / .npy template files.
        seed (int): deterministic seed for random picking.
        pick (str): 'random' (default) or 'first' - selection strategy for the 1 index file.

    Returns:
        index_vectors: np.ndarray (M, D) float32
        index_labels:  np.ndarray (M,) strings like '001_L'
        query_vectors: np.ndarray (Q, D) float32 (empty if none)
        query_labels:  np.ndarray (Q,) strings
    """
    random.seed(seed)
    np.random.seed(seed)

    # regex to extract subject and eye from filename; flexible to variations
    # examples matched: 001_L_01_template.npz, 23_R_3_template.npz, 0001_L_12_template.npz
    pattern = re.compile(r'^(\d+)_([LRlr])_.*template', re.IGNORECASE)

    # collect files
    files = [f for f in os.listdir(root_folder) if f.lower().endswith(('.npz', '.npy'))]
    if not files:
        raise ValueError(f"No .npz/.npy files found in {root_folder}")

    groups = {}  # key -> list of filenames
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            # skip files that don't match pattern (or optionally group them by filename prefix)
            print(f"⚠️ Skipping file with unexpected name format: {f}")
            continue
        subj = m.group(1).zfill(3)  # pad subject id to 3 digits for consistent labels
        eye = m.group(2).upper()
        key = f"{subj}_{eye}"
        groups.setdefault(key, []).append(f)

    # helper to convert a single file into a 1D float32 normalized vector
    def file_to_vector(path):
        full = os.path.join(root_folder, path)
        if full.lower().endswith('.npz'):
            data = np.load(full, allow_pickle=True)
            if 'iris_code' not in data.files or 'mask_code' not in data.files:
                raise ValueError(f"{path} missing 'iris_code' or 'mask_code' keys.")
            iris_code = np.array(data['iris_code']).reshape(-1)
            mask_code = np.array(data['mask_code']).reshape(-1)
            if iris_code.shape != mask_code.shape:
                # try broadcasting or raise
                if iris_code.size == mask_code.size:
                    pass
                else:
                    raise ValueError(f"Shape mismatch in {path}: iris_code {iris_code.shape}, mask_code {mask_code.shape}")
            # mask==1 => occluded/unreliable; set corresponding bits to 0
            vec = np.where(mask_code == 1, 0, iris_code).astype(np.float32)
        elif full.lower().endswith('.npy'):
            vec = np.load(full).reshape(-1).astype(np.float32)
        else:
            raise ValueError(f"Unsupported file: {path}")

        # L2 normalize if possible
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        return vec

    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    # iterate groups
    for key in sorted(groups.keys()):
        group_files = groups[key][:]
        if len(group_files) == 0:
            continue

        if pick == 'random':
            random.shuffle(group_files)
        elif pick == 'first':
            group_files = sorted(group_files)
        else:
            raise ValueError("pick must be 'random' or 'first'")

        # pick first as index, rest are query
        idx_file = group_files[0]
        try:
            idx_vec = file_to_vector(idx_file)
        except Exception as e:
            print(f"⚠️ Failed to load index file {idx_file} for {key}: {e}. Skipping this group.")
            continue

        index_vectors.append(idx_vec)
        index_labels.append(key)

        for qf in group_files[1:]:
            try:
                qvec = file_to_vector(qf)
            except Exception as e:
                print(f"⚠️ Failed to load query file {qf} for {key}: {e}. Skipping this file.")
                continue
            query_vectors.append(qvec)
            query_labels.append(key)

        print(f"{key}: index=1, query={max(0, len(group_files)-1)}")

    if len(index_vectors) == 0:
        raise ValueError("No index vectors found. Check filename patterns and folder contents.")

    index_vectors = np.vstack(index_vectors).astype(np.float32)
    index_labels = np.array(index_labels)

    if len(query_vectors) > 0:
        query_vectors = np.vstack(query_vectors).astype(np.float32)
        query_labels = np.array(query_labels)
    else:
        D = index_vectors.shape[1]
        query_vectors = np.zeros((0, D), dtype=np.float32)
        query_labels = np.array([])

    return index_vectors, index_labels, query_vectors, query_labels


    


In [15]:
data_folder = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Lamp_/outputs_npz/templates"
X_index, y_index, X_query, y_query = load_IrisLamp_templates(data_folder, seed=42)

001_L: index=1, query=19
001_R: index=1, query=19
002_L: index=1, query=19
002_R: index=1, query=19
003_L: index=1, query=19
003_R: index=1, query=19
004_L: index=1, query=19
004_R: index=1, query=19
005_L: index=1, query=19
005_R: index=1, query=19
006_L: index=1, query=19
006_R: index=1, query=19
007_L: index=1, query=19
007_R: index=1, query=19
008_L: index=1, query=19
008_R: index=1, query=19
009_L: index=1, query=19
009_R: index=1, query=19
010_L: index=1, query=19
010_R: index=1, query=19
011_L: index=1, query=19
011_R: index=1, query=19
012_L: index=1, query=19
012_R: index=1, query=19
013_L: index=1, query=19
013_R: index=1, query=19
014_L: index=1, query=19
014_R: index=1, query=19
015_L: index=1, query=19
015_R: index=1, query=19
016_L: index=1, query=19
016_R: index=1, query=19
017_L: index=1, query=19
017_R: index=1, query=19
018_L: index=1, query=19
018_R: index=1, query=19
019_L: index=1, query=19
019_R: index=1, query=19
020_L: index=1, query=19
020_R: index=1, query=19


In [17]:
d = X_index.shape[1]
n_bits = 1024                # tune this (bigger -> finer hashing)
index = faiss.IndexLSH(d, n_bits)
index.add(np.ascontiguousarray(X_index.astype('float32')))

x_labels, hit_rates = evaluate_hit_rate_topk_faiss_lsh(
    faiss_index=index,
    index_labels=y_index,
    query_vectors=X_query,
    query_labels=y_query,
    k_max=100
)


Evaluating top-K hit rates:   5%|▌         | 1/20 [00:03<01:11,  3.78s/it]

K=  1, Hit rate=0.5250


Evaluating top-K hit rates:  10%|█         | 2/20 [00:07<01:07,  3.75s/it]

K=  6, Hit rate=0.6502


Evaluating top-K hit rates:  15%|█▌        | 3/20 [00:11<01:03,  3.76s/it]

...

K= 86, Hit rate=0.8295


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [01:12<00:03,  3.93s/it]

K= 91, Hit rate=0.8339


Evaluating top-K hit rates: 100%|██████████| 20/20 [01:16<00:00,  3.84s/it]

K= 96, Hit rate=0.8373


In [18]:
np.savez('IrisLamp_LSH.npz', x=x_labels, y=hit_rates)

In [17]:
d = X_index.shape[1]
n_bits = 1024                # tune this (bigger -> finer hashing)
index = faiss.IndexLSH(d, n_bits)
index.add(np.ascontiguousarray(X_index.astype('float32')))

x_labels, hit_rates = evaluate_hit_rate_topk_faiss_lsh(
    faiss_index=index,
    index_labels=y_index,
    query_vectors=X_query,
    query_labels=y_query,
    k_max=100
)

plot_hit_rate_plotly_topk(x_labels, hit_rates,23,"CASIAv1 LSH")

Evaluating top-K hit rates:   5%|▌         | 1/20 [00:02<00:45,  2.38s/it]

K=  1, Hit rate=0.5250


Evaluating top-K hit rates:  10%|█         | 2/20 [00:04<00:42,  2.38s/it]

K=  6, Hit rate=0.6502


Evaluating top-K hit rates:  15%|█▌        | 3/20 [00:07<00:40,  2.38s/it]

...

K= 86, Hit rate=0.8295


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [00:47<00:02,  2.53s/it]

K= 91, Hit rate=0.8339


Evaluating top-K hit rates: 100%|██████████| 20/20 [00:49<00:00,  2.48s/it]

K= 96, Hit rate=0.8373


In [18]:
plot_hit_rate_plotly_topk(x_labels, hit_rates,23,"Iris_Lamp LSH")

# Iris Thousand

In [19]:
import os
import re
import random
import numpy as np

def load_IrisThousand_templates(root_folder, seed=42, pick='random', expected_per_group=10):
    """
    Loader for 'Iris Thousand' style filenames like:
      000_L_00_template.npz ... 000_L_09_template.npz
      000_R_00_template.npz ... 000_R_09_template.npz
      ...
      999_R_09_template.npz

    Behavior:
      - Groups files by subject (0..999) and eye (L/R).
      - Picks exactly one file per group as the index vector (strategy: 'random' or 'first').
      - All remaining files in that group become queries for that label.

    Args:
      root_folder (str): path to folder containing .npz / .npy template files.
      seed (int): random seed for deterministic shuffling.
      pick (str): 'random' (default) or 'first' - selection strategy for the 1 index file.
      expected_per_group (int): expected images per subject-eye (default 10). Used to warn if mismatch.

    Returns:
      index_vectors: np.ndarray (M, D) float32
      index_labels:  np.ndarray (M,) strings like '000_L'
      query_vectors: np.ndarray (Q, D) float32 (empty if none)
      query_labels:  np.ndarray (Q,) strings
    """
    random.seed(seed)
    np.random.seed(seed)

    # Pattern extracts subject (1-3 digits), eye (L/R) and image number (00..09 or similar)
    pattern = re.compile(r'^(\d{1,3})_([LRlr])_0*(\d+)_template', re.IGNORECASE)

    files = [f for f in os.listdir(root_folder) if f.lower().endswith(('.npz', '.npy'))]
    if not files:
        raise ValueError(f"No .npz/.npy files found in {root_folder}")

    groups = {}
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            print(f"⚠️ Skipping file with unexpected name format: {f}")
            continue
        subj = m.group(1).zfill(3)   # ensure three-digit subject id like '000'
        eye = m.group(2).upper()
        key = f"{subj}_{eye}"
        groups.setdefault(key, []).append(f)

    def file_to_vector(path):
        full = os.path.join(root_folder, path)
        if full.lower().endswith('.npz'):
            data = np.load(full, allow_pickle=True)
            # keep the iris_code / mask_code logic you already used
            if 'iris_code' not in data.files or 'mask_code' not in data.files:
                raise ValueError(f"{path} missing 'iris_code' or 'mask_code' keys.")
            iris_code = np.array(data['iris_code']).reshape(-1)
            mask_code = np.array(data['mask_code']).reshape(-1)
            if iris_code.shape != mask_code.shape:
                if iris_code.size == mask_code.size:
                    pass
                else:
                    raise ValueError(f"Shape mismatch in {path}: iris_code {iris_code.shape}, mask_code {mask_code.shape}")
            vec = np.where(mask_code == 1, 0, iris_code).astype(np.float32)
        elif full.lower().endswith('.npy'):
            vec = np.load(full).reshape(-1).astype(np.float32)
        else:
            raise ValueError(f"Unsupported file: {path}")

        # L2 normalize
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        return vec

    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    for key in sorted(groups.keys()):
        group_files = groups[key][:]
        if len(group_files) == 0:
            continue

        if len(group_files) != expected_per_group:
            print(f"⚠️ Warning: group {key} has {len(group_files)} files (expected {expected_per_group}).")

        if pick == 'random':
            random.shuffle(group_files)
        elif pick == 'first':
            group_files = sorted(group_files)
        else:
            raise ValueError("pick must be 'random' or 'first'")

        idx_file = group_files[0]
        try:
            idx_vec = file_to_vector(idx_file)
        except Exception as e:
            print(f"⚠️ Failed to load index file {idx_file} for {key}: {e}. Skipping this group.")
            continue

        index_vectors.append(idx_vec)
        index_labels.append(key)

        for qf in group_files[1:]:
            try:
                qvec = file_to_vector(qf)
            except Exception as e:
                print(f"⚠️ Failed to load query file {qf} for {key}: {e}. Skipping this file.")
                continue
            query_vectors.append(qvec)
            query_labels.append(key)

        print(f"{key}: index=1, query={max(0, len(group_files)-1)}")

    if len(index_vectors) == 0:
        raise ValueError("No index vectors found. Check filename patterns and folder contents.")

    index_vectors = np.vstack(index_vectors).astype(np.float32)
    index_labels = np.array(index_labels)

    if len(query_vectors) > 0:
        query_vectors = np.vstack(query_vectors).astype(np.float32)
        query_labels = np.array(query_labels)
    else:
        D = index_vectors.shape[1]
        query_vectors = np.zeros((0, D), dtype=np.float32)
        query_labels = np.array([])

    return index_vectors, index_labels, query_vectors, query_labels


In [20]:
data_folder = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand_/outputs_npz/templates"
X_index, y_index, X_query, y_query = load_IrisThousand_templates(data_folder, seed=42)

⚠️ Warning: group 000_L has 9 files (expected 10).
000_L: index=1, query=8
000_R: index=1, query=9
001_L: index=1, query=9
⚠️ Warning: group 001_R has 9 files (expected 10).
001_R: index=1, query=8
002_L: index=1, query=9
⚠️ Warning: group 002_R has 8 files (expected 10).
002_R: index=1, query=7
003_L: index=1, query=9
003_R: index=1, query=9
004_L: index=1, query=9
004_R: index=1, query=9
005_L: index=1, query=9
005_R: index=1, query=9
⚠️ Warning: group 006_L has 7 files (expected 10).
006_L: index=1, query=6
006_R: index=1, query=9
007_L: index=1, query=9
⚠️ Warning: group 007_R has 7 files (expected 10).
007_R: index=1, query=6
008_L: index=1, query=9
008_R: index=1, query=9
009_L: index=1, query=9
009_R: index=1, query=9
⚠️ Warning: group 010_L has 8 files (expected 10).
010_L: index=1, query=7
⚠️ Warning: group 010_R has 7 files (expected 10).
010_R: index=1, query=6
011_L: index=1, query=9
011_R: index=1, query=9
⚠️ Warning: group 012_L has 9 files (expected 10).
012_L: index=1, 

In [23]:
d = X_index.shape[1]
n_bits = 1024                # tune this (bigger -> finer hashing)
index = faiss.IndexLSH(d, n_bits)
index.add(np.ascontiguousarray(X_index.astype('float32')))

x_labels, hit_rates = evaluate_hit_rate_topk_faiss_lsh(
    faiss_index=index,
    index_labels=y_index,
    query_vectors=X_query,
    query_labels=y_query,
    k_max=200
)

Evaluating top-K hit rates:   5%|▌         | 1/20 [00:03<01:09,  3.66s/it]

K=  1, Hit rate=0.2810


Evaluating top-K hit rates:  10%|█         | 2/20 [00:07<01:05,  3.66s/it]

K= 11, Hit rate=0.4040


Evaluating top-K hit rates:  15%|█▌        | 3/20 [00:11<01:02,  3.68s/it]

...

K=171, Hit rate=0.6175


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [01:14<00:04,  4.14s/it]

K=181, Hit rate=0.6236


Evaluating top-K hit rates: 100%|██████████| 20/20 [01:18<00:00,  3.93s/it]

K=191, Hit rate=0.6306


In [26]:
np.savez('irisThousand_LSH.npz', x=x_labels, y=hit_rates)

In [25]:
plot_hit_rate_plotly_topk(x_labels, hit_rates,23,"Iris_Thousand LSH")